In [77]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torch.optim as optim
import torchvision.datasets as data
import torchvision.transforms as transform

In [78]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [79]:
lr = 0.001
epochs = 10
batch = 64
load_model = True     ## To check if to use checkpoint file or not

In [80]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN, self).__init__()
    self.conv1 = nn.Conv2d(1, 16, (3,3), 1, 1)
    self.pool = nn.MaxPool2d((2,2), 2)
    self.conv2 = nn.Conv2d(16, 32, (3,3), 1, 1)
    self.fc1 = nn.Linear(7*7*32, 10)


  def forward(self, x):
    x = F.relu(self.conv1(x))
    x = self.pool(x)
    x = F.relu(self.conv2(x))
    x = self.pool(x)
    x = x.flatten(1)
    x = self.fc1(x)


    return x

In [81]:
train_data = data.MNIST(root='/data', train=True, download=True, transform=transform.ToTensor())
train_loader = DataLoader(dataset=train_data, shuffle = True, batch_size=batch)

test_data = data.MNIST(root='/data', train=False, download=True, transform=transform.ToTensor())
test_loader = DataLoader(dataset=test_data, shuffle = False, batch_size=batch)

In [82]:
model = CNN().to(device)
loss_func = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

In [83]:
def save_checkpoint(state, filename='/data/checkpoint.pth.tar'):    ## creates a file
  print('-'*32)
  print(f'saving checkpoint')
  torch.save(state, filename)   ## Takes the state dict and saves it on to the file mentioned

In [84]:
def load_checkpoint(checkpoint):
  print('-'*32)
  print('Loading checkpoint')

  model.load_state_dict(checkpoint['state_dict'])     ## loads the parameters from the dict and applies them to the model
  optimizer.load_state_dict(checkpoint['optimizer'])    ## This takes the current state of the optimizer and applies that to the model

if load_model:
  load_checkpoint(torch.load('/data/checkpoint.pth.tar', weights_only=False))

--------------------------------
Loading checkpoint


In [85]:
for i in range(epochs):

  if i % 2 ==0:
    checkpoint = {'state_dict': model.state_dict(), 'optimizer': optimizer.state_dict()}      ## model.state --> learns all the parameters in form or dict of all the layers , optimizer.state --> Learns the current state of the optimizer
    save_checkpoint(checkpoint)

  for batch_idx, (inputs, targets) in enumerate(train_loader):
    inputs = inputs.to(device)
    targets = targets.to(device)

    scores = model(inputs)
    loss = loss_func(scores, targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

--------------------------------
saving checkpoint
--------------------------------
saving checkpoint
--------------------------------
saving checkpoint
--------------------------------
saving checkpoint
--------------------------------
saving checkpoint


In [86]:

def  acc(loader, model):
  correct = 0
  sample = 0
  model.eval()

  with torch.no_grad():
    for x, y in loader:
      x = x.to(device)
      y = y.to(device)

      scores = model(x)
      _, pred = scores.max(1)
      correct += (pred == y).sum()
      sample += pred.size(0)

  model.train()

  return correct, sample

train_correct, train_samples = acc(train_loader, model)
print(f"Accuracy on training set: {train_correct.item() / train_samples * 100:.2f}%")

test_correct, test_samples = acc(test_loader, model)
print(f"Accuracy on test set: {test_correct.item() / test_samples * 100:.2f}%")


Accuracy on training set: 99.87%
Accuracy on test set: 98.98%
